# Notebook 00 — Preparación de datos

*Alimenta el apartado 2 de la memoria. Sus bloques no se corresponden con
subapartados concretos, por tratarse de un paso de conversión previo.*

Conversión del conjunto de datos `idealista18` desde su formato original de R a
formatos legibles en Python.

Este notebook se ejecuta **una sola vez**: su resultado son los ficheros
almacenados en `data/raw`, que constituyen el punto de partida inmutable del
análisis. Es el único punto del trabajo en el que interviene R; el resto del TFM
se desarrolla íntegramente en Python.

## 1. Configuración del entorno de trabajo

El entorno de ejecución de Google Colab es efímero: la máquina virtual se
destruye al cerrar la sesión y todo lo almacenado en su disco local se pierde.
Para que el conjunto de datos convertido persista entre sesiones, se monta
Google Drive como sistema de almacenamiento y se define en él la estructura de
carpetas del proyecto.

La ruta base se declara en una única constante (`BASE`), de forma que el
notebook siga siendo válido si el proyecto cambia de ubicación.

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import os

BASE = '/content/drive/MyDrive/Master Data Science/TFM_AVM'

if not os.path.exists(BASE):
    raise FileNotFoundError(
        f'No existe {BASE}. Carpetas disponibles: '
        f'{os.listdir("/content/drive/MyDrive")}'
    )

for carpeta in ['data/raw', 'data/processed', 'figuras', 'modelos']:
    os.makedirs(f'{BASE}/{carpeta}', exist_ok=True)
    print(f'{carpeta}/  listo')

data/raw/  listo
data/processed/  listo
figuras/  listo
modelos/  listo


## 2. Instalación de dependencias

El conjunto de datos se distribuye como un paquete de R (`idealista18`), no
disponible en CRAN, por lo que se instala directamente desde su repositorio de
GitHub.

Se requiere además el paquete `sf` (*simple features*), que implementa el
estándar ISO/OGC de representación de geometrías y es el responsable de la
escritura en formato GeoPackage. Al tratarse de una interfaz sobre librerías de
sistema (GDAL, GEOS y PROJ), se instala como binario precompilado mediante
`apt-get` para evitar su compilación desde código fuente.

La extensión `rpy2.ipython` permite ejecutar celdas de R (`%%R`) dentro de un
notebook de Python.

El equivalente de `sf` en el ecosistema Python es `geopandas`; ambos se apoyan en
las mismas librerías subyacentes, lo que garantiza que la conversión no
introduzca pérdidas.

In [42]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [43]:
%%R
# La versión se fija de forma explícita: sin `ref`, install_github instala la
# rama principal y una reejecución futura podría obtener un paquete distinto.
# El repositorio no publica etiquetas, de modo que se fija el hash del commit
# correspondiente a la versión 0.1.2, que es la empleada en este trabajo.
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
remotes::install_github("paezha/idealista18",
                        ref = "e9ef80618f8a9cc28f672ac138f298bbd519918e")

Skipping install of 'idealista18' from a github remote, the SHA1 (e9ef8061) has not changed since last install.
  Use `force = TRUE` to force installation


In [44]:
%%R
library(idealista18)
cat("idealista18 versión:", as.character(packageVersion("idealista18")), "\n")
stopifnot(as.character(packageVersion("idealista18")) == "0.1.2")

idealista18 versión: 0.1.2 


In [45]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y r-cran-sf > /dev/null 2>&1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [46]:
%%R
if (requireNamespace("sf", quietly = TRUE)) {
  library(sf)
  cat("sf OK, versión:", as.character(packageVersion("sf")), "\n")
} else {
  cat("sf NO disponible\n")
}

sf OK, versión: 1.1.3 


## 3. Inventario de los objetos del paquete

Se comprueban los objetos disponibles, sus dimensiones y el sistema de
referencia de coordenadas.

El paquete contiene diez objetos: para cada una de las tres ciudades, los
anuncios de venta (`*_Sale`), los polígonos de barrio (`*_Polygons`) y los
puntos de interés (`*_POIS`); más un objeto adicional agregado por distrito.

El CRS resulta ser **EPSG:4326** (WGS84), es decir, coordenadas expresadas en
grados. Este dato condiciona la fase posterior de enriquecimiento geoespacial:
al no ser el grado una unidad de distancia constante, será necesario reproyectar
a un sistema métrico antes de calcular distancias.

In [47]:
%%R
library(idealista18)
library(sf)

objetos <- data(package = "idealista18")$results[, "Item"]
print(objetos)

cat("\nCRS:", sf::st_crs(Madrid_Sale)$input, "\n")
cat("Dimensiones Madrid_Sale:", dim(Madrid_Sale), "\n")

 [1] "Barcelona_POIS"         "Barcelona_Polygons"     "Barcelona_Sale"        
 [4] "Madrid_POIS"            "Madrid_Polygons"        "Madrid_Sale"           
 [7] "properties_by_district" "Valencia_POIS"          "Valencia_Polygons"     
[10] "Valencia_Sale"         

CRS: EPSG:4326 
Dimensiones Madrid_Sale: 94815 42 


## 4. Conversión a formatos legibles desde Python

Los objetos de R se almacenan en formato binario propio (`.rda`) y contienen una
columna de geometría que no admite representación tabular plana. Se exportan a
GeoPackage (`.gpkg`), formato estándar abierto que preserva geometría y CRS y es
legible desde Python mediante `geopandas`. Esta conversión se ejecuta una sola
vez.

Los objetos de puntos de interés (`*_POIS`) no son entidades espaciales sino
listas de tablas planas con columnas `Lon`/`Lat`. La celda 4.2 los omite
deliberadamente —de ahí el aviso `OMITIDO (clase: list)`— y se exportan a CSV en
la celda 4.3, formato suficiente para su estructura.

### 4.1. Carpeta de trabajo temporal

In [48]:
import os
os.makedirs('/content/data_raw', exist_ok=True)
print(os.path.exists('/content/data_raw'))

True


### 4.2. Exportación de los objetos espaciales

In [49]:
%%R
library(idealista18)
library(sf)

destino <- "/content/data_raw"
objetos <- data(package = "idealista18")$results[, "Item"]

for (nm in objetos) {
  obj <- get(nm)
  res <- tryCatch({
    if (inherits(obj, "sf")) {
      ruta <- file.path(destino, paste0(nm, ".gpkg"))
      st_write(obj, ruta, delete_dsn = TRUE, quiet = TRUE)
      sprintf("%7d filas -> %s", nrow(obj), basename(ruta))
    } else if (is.data.frame(obj)) {
      ruta <- file.path(destino, paste0(nm, ".csv"))
      write.csv(obj, ruta, row.names = FALSE)
      sprintf("%7d filas -> %s", nrow(obj), basename(ruta))
    } else {
      sprintf("OMITIDO (clase: %s)", paste(class(obj), collapse=", "))
    }
  }, error = function(e) paste("ERROR:", conditionMessage(e)))
  cat(sprintf("%-24s %s\n", nm, res))
}

Barcelona_POIS           OMITIDO (clase: list)
Barcelona_Polygons            69 filas -> Barcelona_Polygons.gpkg
Barcelona_Sale             61486 filas -> Barcelona_Sale.gpkg
Madrid_POIS              OMITIDO (clase: list)
Madrid_Polygons              135 filas -> Madrid_Polygons.gpkg
Madrid_Sale                94815 filas -> Madrid_Sale.gpkg
properties_by_district        47 filas -> properties_by_district.gpkg
Valencia_POIS            OMITIDO (clase: list)
Valencia_Polygons             73 filas -> Valencia_Polygons.gpkg
Valencia_Sale              33622 filas -> Valencia_Sale.gpkg


### 4.3. Exportación de los puntos de interés

In [50]:
%%R
destino <- "/content/data_raw"

for (ciudad in c("Madrid", "Barcelona", "Valencia")) {
  obj <- get(paste0(ciudad, "_POIS"))
  for (k in names(obj)) {
    nombre <- paste0(ciudad, "_POIS_", k, ".csv")
    write.csv(obj[[k]], file.path(destino, nombre), row.names = FALSE)
    cat(sprintf("%-36s %5d filas\n", nombre, nrow(obj[[k]])))
  }
}

Madrid_POIS_City_Center.csv              1 filas
Madrid_POIS_Metro.csv                  240 filas
Madrid_POIS_Castellana.csv             155 filas
Barcelona_POIS_City_Center.csv           1 filas
Barcelona_POIS_Metro.csv               463 filas
Barcelona_POIS_Diagonal.csv             57 filas
Valencia_POIS_City_Center.csv            1 filas
Valencia_POIS_Metro.csv                 99 filas
Valencia_POIS_Blasco.csv                27 filas


### 4.4. Copia al almacenamiento persistente

In [51]:
import shutil
from pathlib import Path

ORIGEN = Path('/content/data_raw')
DESTINO = Path(BASE) / 'data/raw'
DESTINO.mkdir(parents=True, exist_ok=True)

generados = sorted(f for f in ORIGEN.iterdir() if f.is_file())
for f in generados:
    shutil.copy(f, DESTINO / f.name)
    print(f'{f.name:36s} {f.stat().st_size / 1e6:7.2f} MB')

gpkg = sum(f.suffix == '.gpkg' for f in generados)
print(f'\nGenerados por este notebook: {len(generados)} ficheros '
      f'({gpkg} GeoPackage y {len(generados) - gpkg} CSV)')
print(f'Copiados a data/raw/, que puede contener además fuentes externas '
      f'incorporadas en notebooks posteriores.')

Barcelona_POIS_City_Center.csv          0.00 MB
Barcelona_POIS_Diagonal.csv             0.00 MB
Barcelona_POIS_Metro.csv                0.01 MB
Barcelona_Polygons.gpkg                 0.17 MB
Barcelona_Sale.gpkg                    13.88 MB
Madrid_POIS_Castellana.csv              0.00 MB
Madrid_POIS_City_Center.csv             0.00 MB
Madrid_POIS_Metro.csv                   0.01 MB
Madrid_Polygons.gpkg                    0.24 MB
Madrid_Sale.gpkg                       21.36 MB
Valencia_POIS_Blasco.csv                0.00 MB
Valencia_POIS_City_Center.csv           0.00 MB
Valencia_POIS_Metro.csv                 0.00 MB
Valencia_Polygons.gpkg                  0.16 MB
Valencia_Sale.gpkg                      7.63 MB
properties_by_district.gpkg             0.33 MB

Generados por este notebook: 16 ficheros (7 GeoPackage y 9 CSV)
Copiados a data/raw/, que puede contener además fuentes externas incorporadas en notebooks posteriores.


## 5. Verificación

Se comprueba desde Python que los ficheros generados son legibles y que la
conversión ha preservado la información original. La verificación detallada
—dimensiones, sistema de referencia, tipo de geometría y nomenclatura de las
variables— se practica sobre Madrid, ámbito del trabajo, y se completa con el
recuento de anuncios de las tres ciudades.

In [52]:
import geopandas as gpd
import pandas as pd

madrid = gpd.read_file(f'{BASE}/data/raw/Madrid_Sale.gpkg')
print('Dimensiones:', madrid.shape)
print('CRS:', madrid.crs)
print('Geometría:', madrid.geometry.geom_type.unique())

metro = pd.read_csv(f'{BASE}/data/raw/Madrid_POIS_Metro.csv')
print('\nMetro:', metro.shape, '| columnas:', list(metro.columns))

print('\nColumnas de Madrid_Sale:')
print(list(madrid.columns))

# Recuento de anuncios de venta, sin cargar la geometría
total = 0
print('\nAnuncios de venta por ciudad')
for ciudad in ['Madrid', 'Barcelona', 'Valencia']:
    n = len(gpd.read_file(f'{BASE}/data/raw/{ciudad}_Sale.gpkg',
                          columns=['ASSETID'], ignore_geometry=True))
    total += n
    print(f'  {ciudad:10s} {n:>7,}')
print(f'  {"Total":10s} {total:>7,}')

Dimensiones: (94815, 42)
CRS: EPSG:4326
Geometría: ['Point']

Metro: (240, 2) | columnas: ['Lon', 'Lat']

Columnas de Madrid_Sale:
['ASSETID', 'PERIOD', 'PRICE', 'UNITPRICE', 'CONSTRUCTEDAREA', 'ROOMNUMBER', 'BATHNUMBER', 'HASTERRACE', 'HASLIFT', 'HASAIRCONDITIONING', 'AMENITYID', 'HASPARKINGSPACE', 'ISPARKINGSPACEINCLUDEDINPRICE', 'PARKINGSPACEPRICE', 'HASNORTHORIENTATION', 'HASSOUTHORIENTATION', 'HASEASTORIENTATION', 'HASWESTORIENTATION', 'HASBOXROOM', 'HASWARDROBE', 'HASSWIMMINGPOOL', 'HASDOORMAN', 'HASGARDEN', 'ISDUPLEX', 'ISSTUDIO', 'ISINTOPFLOOR', 'CONSTRUCTIONYEAR', 'FLOORCLEAN', 'FLATLOCATIONID', 'CADCONSTRUCTIONYEAR', 'CADMAXBUILDINGFLOOR', 'CADDWELLINGCOUNT', 'CADASTRALQUALITYID', 'BUILTTYPEID_1', 'BUILTTYPEID_2', 'BUILTTYPEID_3', 'DISTANCE_TO_CITY_CENTER', 'DISTANCE_TO_METRO', 'DISTANCE_TO_CASTELLANA', 'LONGITUDE', 'LATITUDE', 'geometry']

Anuncios de venta por ciudad
  Madrid      94,815
  Barcelona   61,486
  Valencia    33,622
  Total      189,923


---

**Nota sobre las variables de distancia.** El conjunto incluye ya tres variables
de distancia precalculadas (`DISTANCE_TO_CITY_CENTER`, `DISTANCE_TO_METRO` y
`DISTANCE_TO_CASTELLANA`). El enriquecimiento geoespacial desarrollado en el
notebook siguiente se orienta, por tanto, a la construcción de variables que el
conjunto original no proporciona, y no a la réplica de las existentes.